# RNA-seq 仿真数据生成与矩阵分析

本 Notebook 会先安装缺失的 R / Bioconductor 包，然后生成模拟 RNA-seq count matrix 和样本信息，再进行 QC、DESeq2 差异分析、PCA、火山图和 GO 富集分析。


In [ ]:
# 安装本 Notebook 需要的 R 包。已经安装过的包会自动跳过。
options(repos = c(CRAN = "https://cloud.r-project.org"))

cran_pkgs <- c("dplyr", "tibble", "tidyr", "ggplot2", "ggrepel", "pheatmap")
bioc_pkgs <- c("DESeq2", "clusterProfiler", "org.Hs.eg.db", "AnnotationDbi", "enrichplot")

missing_cran <- cran_pkgs[!vapply(cran_pkgs, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing_cran) > 0) {
  install.packages(missing_cran)
}

if (!requireNamespace("BiocManager", quietly = TRUE)) {
  install.packages("BiocManager")
}

missing_bioc <- bioc_pkgs[!vapply(bioc_pkgs, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing_bioc) > 0) {
  BiocManager::install(missing_bioc, ask = FALSE, update = FALSE)
}


## 0. 设置工作目录

Colab 中默认使用 `/content/rnaseq_matrix_demo`；其他 Jupyter 环境中默认在当前目录下创建 `rnaseq_matrix_demo`。


In [ ]:
workdir <- if (dir.exists("/content")) {
  "/content/rnaseq_matrix_demo"
} else {
  file.path(getwd(), "rnaseq_matrix_demo")
}

outdir <- workdir
result_dir <- file.path(workdir, "teaching_results")
figure_dir <- file.path(workdir, "teaching_figures")

dir.create(workdir, recursive = TRUE, showWarnings = FALSE)
dir.create(result_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(figure_dir, recursive = TRUE, showWarnings = FALSE)

workdir


# 一、生成仿真 RNA-seq count matrix


In [ ]:
suppressPackageStartupMessages({
  library(org.Hs.eg.db)
  library(AnnotationDbi)
  library(dplyr)
})

set.seed(20260521)

outdir

In [ ]:
all_symbols <- keys(org.Hs.eg.db, keytype = "SYMBOL")

# 保留格式比较规范的 gene symbol
all_symbols <- all_symbols[
  grepl("^[A-Za-z0-9.-]+$", all_symbols)
]

# 去重
all_symbols <- unique(all_symbols)

length(all_symbols)
head(all_symbols)

In [ ]:
up_genes <- c(
  "IL1B", "IL6", "TNF", "CXCL8", "CXCL10", "CCL2", "CCL3", "CCL4",
  "NFKB1", "NFKBIA", "RELA", "STAT1", "STAT3", "IRF1", "IRF7",
  "ISG15", "IFI6", "IFI44", "IFIT1", "IFIT2", "IFIT3", "MX1",
  "OAS1", "OAS2", "HLA-A", "HLA-B", "HLA-C", "HLA-DRA",
  "HLA-DRB1", "CD74", "S100A8", "S100A9", "LYZ", "LST1",
  "HMOX1", "VEGFA"
)

down_genes <- c(
  "CD3D", "CD3E", "CD4", "CD8A", "CD8B",
  "MS4A1", "CD79A", "CD79B",
  "NKG7", "GNLY", "GZMB", "PRF1",
  "MKI67", "TOP2A", "PCNA", "MCM2", "MCM3", "MCM4",
  "MCM5", "MCM6", "CDK1", "CCNB1", "CCNB2", "AURKA", "AURKB"
)

# 确保这些基因都在 org.Hs.eg.db 中
up_genes <- intersect(up_genes, all_symbols)
down_genes <- intersect(down_genes, all_symbols)

length(up_genes)
length(down_genes)

In [ ]:
n_total_genes <- 3000

de_genes <- unique(c(up_genes, down_genes))

background_genes <- setdiff(all_symbols, de_genes)
background_genes <- sample(background_genes, n_total_genes - length(de_genes))

genes <- unique(c(de_genes, background_genes))

length(genes)
head(genes)

In [ ]:
sample_info <- data.frame(
  sample_id = c(paste0("control_", 1:3), paste0("disease_", 1:3)),
  group = c(rep("control", 3), rep("disease", 3)),
  stringsAsFactors = FALSE
)

rownames(sample_info) <- sample_info$sample_id

sample_info

In [ ]:
n_genes <- length(genes)
n_samples <- nrow(sample_info)

# 基因基础表达量：log-normal 分布，模拟不同基因表达量跨度
base_mean <- rlnorm(n_genes, meanlog = log(100), sdlog = 1.2)
names(base_mean) <- genes
base_mean[base_mean < 1] <- 1

# 样本测序深度差异
lib_size_factor <- c(0.95, 1.05, 1.00, 1.10, 0.90, 1.05)
names(lib_size_factor) <- sample_info$sample_id

# 设置 disease/control 的真实 fold change
fold_change <- rep(1, n_genes)
names(fold_change) <- genes

fold_change[up_genes] <- 4       # disease 上调，约 log2FC = 2
fold_change[down_genes] <- 0.25  # disease 下调，约 log2FC = -2

# Negative binomial dispersion
dispersion <- 0.15
size <- 1 / dispersion

count_mat <- matrix(
  0,
  nrow = n_genes,
  ncol = n_samples,
  dimnames = list(genes, sample_info$sample_id)
)

for (gene in genes) {
  for (sample in sample_info$sample_id) {
    group <- sample_info[sample, "group"]
    
    mu <- base_mean[gene] * lib_size_factor[sample]
    
    if (group == "disease") {
      mu <- mu * fold_change[gene]
    }
    
    count_mat[gene, sample] <- rnbinom(1, mu = mu, size = size)
  }
}

# 过滤极低表达基因
count_mat <- count_mat[rowSums(count_mat) >= 10, ]

dim(count_mat)
head(count_mat)

In [ ]:
truth_de <- data.frame(
  gene = c(up_genes, down_genes),
  true_direction = c(
    rep("up_in_disease", length(up_genes)),
    rep("down_in_disease", length(down_genes))
  ),
  stringsAsFactors = FALSE
)

write.table(
  count_mat,
  file = file.path(outdir, "count_matrix.tsv"),
  sep = "\t",
  quote = FALSE,
  col.names = NA
)

write.table(
  sample_info,
  file = file.path(outdir, "sample_info.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

write.table(
  truth_de,
  file = file.path(outdir, "truth_DE_genes.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

list.files(outdir)

# 二、RNA-seq 矩阵形式数据分析


# RNA-seq 表达矩阵分析教程

本教程包含以下内容：

1. 读取表达矩阵和样本分组信息；
2. 对样本进行基础质检，包括总 counts、检出基因数和表达分布；
3. 使用 DESeq2 构建差异分析对象；
4. 进行 VST 标准化转换；
5. 绘制 PCA 图，检查样本整体分布；
6. 进行 disease vs control 差异表达分析；
7. 绘制火山图；
8. 对上调和下调基因分别进行 GO Biological Process 富集分析。

> 注意：DESeq2 的输入应为 **raw count matrix**，不建议直接使用 TPM、FPKM 或 CPM 作为 DESeq2 输入。


## 0. 输入文件说明

本教程需要两个输入文件：

| 文件 | 说明 |
|---|---|
| `count_matrix.tsv` | RNA-seq count matrix，行为基因，列为样本，数值为原始 read count |
| `sample_info.tsv` | 样本信息表，至少包含 `sample_id` 和 `group` 两列 |

矩阵格式示例：

| gene | control_1 | control_2 | disease_1 |
|---|---:|---:|---:|
| IL6 | 20 | 24 | 150 |
| TNF | 30 | 28 | 180 |

样本信息表示例：

| sample_id | group |
|---|---|
| control_1 | control |
| disease_1 | disease |


## 1. 加载 R 包

这一部分加载后续分析需要的 R 包。

- `DESeq2`：用于 RNA-seq count matrix 的标准化和差异表达分析；
- `ggplot2` / `ggrepel`：用于绘图和标签标注；
- `clusterProfiler`：用于 GO 富集分析；
- `org.Hs.eg.db`：人类基因注释数据库，用于 gene symbol 和 ENTREZ ID 转换；
- `dplyr` / `tibble` / `tidyr`：用于数据整理。


In [ ]:
suppressPackageStartupMessages({
  library(DESeq2)
  library(dplyr)
  library(tibble)
  library(tidyr)
  library(ggplot2)
  library(ggrepel)
  library(clusterProfiler)
  library(org.Hs.eg.db)
  library(AnnotationDbi)
  library(enrichplot)
})

# 设置随机种子，保证部分需要随机过程的分析结果可复现
set.seed(20260521)


## 2. 设置输入和输出路径

这里统一设置工作目录、输入文件路径、结果目录和图片目录。后续所有结果都会规范保存到对应目录中，方便课程演示和结果整理。


In [ ]:
# 工作目录：存放输入矩阵、样本信息、输出结果和图片
if (!exists("workdir")) {
  workdir <- if (dir.exists("/content")) "/content/rnaseq_matrix_demo" else file.path(getwd(), "rnaseq_matrix_demo")
}

# 输入文件
count_file <- file.path(workdir, "count_matrix.tsv")
sample_file <- file.path(workdir, "sample_info.tsv")

# 输出目录
result_dir <- file.path(workdir, "teaching_results")
figure_dir <- file.path(workdir, "teaching_figures")

# 如果输出目录不存在，则自动创建
# recursive = TRUE 表示可以递归创建多层目录
# showWarnings = FALSE 表示目录已存在时不报 warning
dir.create(result_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(figure_dir, recursive = TRUE, showWarnings = FALSE)

# 打印输入文件路径，方便检查
count_file
sample_file


## 3. 读取 count matrix 和样本信息

这一节是矩阵分析最关键的基础步骤。

需要重点检查：

1. count matrix 的列名是否是样本名；
2. `sample_info$sample_id` 是否与 count matrix 的列名一致；
3. 分组变量 `group` 是否正确设置了对照组和疾病组的顺序。

在本教程中，`control` 被设置为参考水平，因此后续 `disease vs control` 的 `log2FoldChange > 0` 表示 disease 组表达更高。


In [ ]:
# 读取 count matrix
# row.names = 1 表示第一列作为基因名
# check.names = FALSE 表示不要自动修改样本名，例如不要把 '-' 改成 '.'
count_mat <- read.delim(
  count_file,
  row.names = 1,
  check.names = FALSE
)

# 读取样本信息表
sample_info <- read.delim(
  sample_file,
  check.names = FALSE
)

# 将 sample_id 设置为 sample_info 的行名，方便按照样本名索引
rownames(sample_info) <- sample_info$sample_id

# 确保 count matrix 的列顺序与 sample_info 的样本顺序完全一致
# 这一步非常重要，否则样本表达量和分组信息可能错位
count_mat <- count_mat[, sample_info$sample_id]

# 设置分组变量，并指定 control 为参考水平
# 因此后续 disease vs control 的 log2FC > 0 表示 disease 组上调
sample_info$group <- factor(
  sample_info$group,
  levels = c("control", "disease")
)

# 基础检查
cat("Count matrix dimension:\n")
print(dim(count_mat))

cat("\nFirst few rows of count matrix:\n")
print(head(count_mat[, 1:min(4, ncol(count_mat))]))

cat("\nSample information:\n")
print(sample_info)


## 4. 样本基础质检：总 reads 数和检出基因数

在进入差异分析前，需要先检查每个样本的基本数据量。

这里计算两个常用指标：

| 指标 | 含义 |
|---|---|
| `total_counts` | 每个样本所有基因 count 的总和，可近似反映测序深度 |
| `detected_genes` | 每个样本中 count > 0 的基因数量，可反映基因检出情况 |

如果某个样本的总 counts 或检出基因数明显低于其他样本，可能需要进一步检查数据质量。


In [ ]:
# 计算每个样本的总 counts 和检出基因数
qc_summary <- data.frame(
  sample_id = colnames(count_mat),
  group = sample_info[colnames(count_mat), "group"],
  total_counts = colSums(count_mat),
  detected_genes = colSums(count_mat > 0),
  stringsAsFactors = FALSE
)

# 展示样本质控汇总表
qc_summary

# 保存质控结果表，便于后续写报告
write.table(
  qc_summary,
  file = file.path(result_dir, "sample_qc_summary.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)


## 5. 绘制样本总 counts 柱状图

该图用于观察不同样本的测序深度是否大致接近。真实项目中，如果某个样本 total counts 远低于其他样本，需要谨慎判断是否为低质量样本。


In [ ]:
p_library_size <- ggplot(qc_summary, aes(x = sample_id, y = total_counts, fill = group)) +
  geom_col(width = 0.7) +
  theme_bw(base_size = 13) +
  labs(
    title = "Library size of each sample",
    x = "Sample",
    y = "Total counts"
  ) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    panel.grid = element_blank()
  )

p_library_size

# 保存图片
# PDF 适合论文和课件编辑，PNG 适合直接插入文档或网页展示
ggsave(file.path(figure_dir, "01_library_size.pdf"), p_library_size, width = 6, height = 4)
ggsave(file.path(figure_dir, "01_library_size.png"), p_library_size, width = 6, height = 4, dpi = 300)


## 6. 绘制检出基因数柱状图

该图用于检查不同样本中被检测到的基因数量是否明显异常。对于同一批次、同一数据类型的 RNA-seq 样本，检出基因数通常不应出现极端离群。


In [ ]:
p_detected_genes <- ggplot(qc_summary, aes(x = sample_id, y = detected_genes, fill = group)) +
  geom_col(width = 0.7) +
  theme_bw(base_size = 13) +
  labs(
    title = "Detected genes of each sample",
    x = "Sample",
    y = "Number of detected genes"
  ) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    panel.grid = element_blank()
  )

p_detected_genes

# 保存图片
ggsave(file.path(figure_dir, "02_detected_genes.pdf"), p_detected_genes, width = 6, height = 4)
ggsave(file.path(figure_dir, "02_detected_genes.png"), p_detected_genes, width = 6, height = 4, dpi = 300)


## 7. 构建 DESeq2 对象

DESeq2 需要两个核心输入：

1. `countData`：原始 count matrix；
2. `colData`：样本信息表；
3. `design`：实验设计公式。

这里使用 `design = ~ group`，表示我们关心的主要因素是 disease/control 分组。


In [ ]:
# 构建 DESeq2 对象
# 注意：countData 必须是整数 count，不能是 TPM/FPKM
# round() 是为了防止读取或处理后出现非整数，但真实分析中应尽量使用原始整数 counts
dds <- DESeqDataSetFromMatrix(
  countData = round(as.matrix(count_mat)),
  colData = sample_info,
  design = ~ group
)

dds


## 8. 过滤低表达基因

低表达基因通常统计信息不足，容易增加多重检验负担。这里过滤掉所有样本总 counts 小于 10 的基因。

教学中可以强调：过滤阈值不是固定不变的，需要根据数据规模、测序深度和研究目的调整。


In [ ]:
# 保留所有样本中总 count >= 10 的基因
keep <- rowSums(counts(dds)) >= 10
dds <- dds[keep, ]

dds


## 9. DESeq2 标准化和 VST 转换

`DESeq()` 会完成 size factor 估计、离散度估计和差异分析模型拟合。

`vst()` 是 variance stabilizing transformation，用于将 count 数据转换为更适合 PCA、聚类和可视化的表达矩阵。

注意：

- 差异分析使用 DESeq2 模型结果；
- PCA 和表达分布图通常使用 VST 或 rlog 转换后的矩阵。


In [ ]:
# 运行 DESeq2 主流程
dds <- DESeq(dds)

# VST 转换，用于后续 PCA 和表达分布可视化
# blind = FALSE 表示转换时考虑实验设计，更适合已有明确分组的分析
vsd <- vst(dds, blind = FALSE)

# 提取 VST 标准化后的表达矩阵
vst_mat <- assay(vsd)

cat("VST matrix dimension:\n")
print(dim(vst_mat))

cat("\nFirst few rows of VST matrix:\n")
print(head(vst_mat[, 1:min(4, ncol(vst_mat))]))

# 保存 VST 矩阵
write.table(
  vst_mat,
  file = file.path(result_dir, "vst_normalized_matrix.tsv"),
  sep = "\t",
  quote = FALSE,
  col.names = NA
)


## 10. 表达分布检查：VST 后箱线图

箱线图用于检查不同样本经过标准化转换后的表达分布是否一致。如果某个样本整体分布明显偏移，可能提示样本质量、批次效应或其他技术因素。


In [ ]:
# 将矩阵转换成长表格式，便于 ggplot2 绘图
vst_long <- as.data.frame(vst_mat) %>%
  rownames_to_column("gene") %>%
  pivot_longer(
    cols = -gene,
    names_to = "sample_id",
    values_to = "expression"
  ) %>%
  left_join(sample_info, by = "sample_id")

p_boxplot <- ggplot(vst_long, aes(x = sample_id, y = expression, fill = group)) +
  geom_boxplot(outlier.size = 0.2) +
  theme_bw(base_size = 13) +
  labs(
    title = "Expression distribution after VST normalization",
    x = "Sample",
    y = "VST expression"
  ) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    panel.grid = element_blank()
  )

p_boxplot

# 保存图片
ggsave(file.path(figure_dir, "03_vst_expression_boxplot.pdf"), p_boxplot, width = 7, height = 4)
ggsave(file.path(figure_dir, "03_vst_expression_boxplot.png"), p_boxplot, width = 7, height = 4, dpi = 300)


## 11. PCA 分析

PCA 用于观察样本整体表达模式。这里对 VST 转换后的矩阵进行 PCA。

注意：

- 矩阵原本是 gene × sample；
- PCA 希望每一行是一个样本，因此需要 `t(vst_mat)` 转置；
- 如果 disease 和 control 能明显分开，说明组间整体表达模式存在差异。


In [ ]:
# 对样本进行 PCA，因此需要将 gene × sample 转为 sample × gene
pca <- prcomp(t(vst_mat), scale. = FALSE)

# 计算每个主成分解释的方差比例
pca_var <- pca$sdev^2 / sum(pca$sdev^2)

# 整理 PCA 结果表
pca_df <- data.frame(
  sample_id = rownames(pca$x),
  PC1 = pca$x[, 1],
  PC2 = pca$x[, 2],
  group = sample_info[rownames(pca$x), "group"],
  stringsAsFactors = FALSE
)

pca_df


## 12. 绘制 PCA 图

PCA 图是 RNA-seq 矩阵分析中常见的样本层面质控图。它可以帮助我们判断：

1. 同组样本是否聚在一起；
2. 不同组样本是否分开；
3. 是否存在明显离群样本。


In [ ]:
p_pca <- ggplot(pca_df, aes(x = PC1, y = PC2, color = group, label = sample_id)) +
  geom_point(size = 4) +
  geom_text_repel(size = 3.5, max.overlaps = 20) +
  theme_bw(base_size = 13) +
  labs(
    title = "PCA based on VST-normalized expression matrix",
    x = paste0("PC1: ", round(pca_var[1] * 100, 1), "%"),
    y = paste0("PC2: ", round(pca_var[2] * 100, 1), "%")
  ) +
  theme(
    panel.grid = element_blank(),
    plot.title = element_text(hjust = 0.5)
  )

p_pca

# 保存图片
ggsave(file.path(figure_dir, "04_PCA.pdf"), p_pca, width = 6, height = 5)
ggsave(file.path(figure_dir, "04_PCA.png"), p_pca, width = 6, height = 5, dpi = 300)


## 13. 差异表达分析：disease vs control

这里使用 `results()` 提取 disease 相对于 control 的差异分析结果。

结果中几个重要字段：

| 字段 | 含义 |
|---|---|
| `baseMean` | 基因在所有样本中的平均标准化表达量 |
| `log2FoldChange` | disease 相对于 control 的 log2 倍数变化 |
| `lfcSE` | log2FoldChange 的标准误 |
| `stat` | Wald 检验统计量 |
| `pvalue` | 原始 P 值 |
| `padj` | 多重检验校正后的 P 值 |

在本教程中，差异基因筛选标准为：`padj < 0.05` 且 `|log2FoldChange| >= 1`。


In [ ]:
# 提取 disease vs control 的差异分析结果
# 因为 group 的 levels 是 c("control", "disease")，所以这里表示 disease / control
res <- results(
  dds,
  contrast = c("group", "disease", "control")
)

# 转换为 data.frame，并把基因名从行名转成一列
res_df <- as.data.frame(res) %>%
  rownames_to_column("gene") %>%
  arrange(padj)

# 根据 padj 和 log2FoldChange 标记上调、下调和非显著基因
res_df <- res_df %>%
  mutate(
    regulation = case_when(
      !is.na(padj) & padj < 0.05 & log2FoldChange >= 1 ~ "Up",
      !is.na(padj) & padj < 0.05 & log2FoldChange <= -1 ~ "Down",
      TRUE ~ "Not significant"
    )
  )

cat("Top differential expression results:\n")
print(head(res_df))

cat("\nDEG summary:\n")
print(table(res_df$regulation))


## 14. 保存差异分析结果

这里分别保存：

1. 全部基因的差异分析结果；
2. 显著差异基因结果。

建议教学时强调：完整结果表应保留，不能只保存显著基因，因为后续复查、GSEA 或阈值调整都可能需要完整结果。


In [ ]:
# 保存全部基因差异分析结果
write.table(
  res_df,
  file = file.path(result_dir, "DESeq2_disease_vs_control_all.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

# 筛选显著差异基因
deg_df <- res_df %>%
  filter(!is.na(padj), padj < 0.05, abs(log2FoldChange) >= 1)

# 保存显著差异基因
write.table(
  deg_df,
  file = file.path(result_dir, "DESeq2_disease_vs_control_DEG.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

cat("DEG dimension:\n")
print(dim(deg_df))

cat("\nTop DEGs:\n")
print(head(deg_df))


## 15. 绘制火山图

火山图同时展示差异倍数和统计显著性：

- 横轴：`log2FoldChange`；
- 纵轴：`-log10(padj)`；
- 右侧为 disease 中上调基因；
- 左侧为 disease 中下调基因。

火山图适合快速展示差异表达分析的整体结果。


In [ ]:
# 计算 -log10(padj)
# padj 越小，-log10(padj) 越大，图上位置越高
volcano_df <- res_df %>%
  mutate(
    neg_log10_padj = -log10(padj),
    neg_log10_padj = ifelse(is.infinite(neg_log10_padj), NA, neg_log10_padj)
  )

# 选择最显著的前 15 个差异基因进行标注
label_df <- volcano_df %>%
  filter(regulation != "Not significant") %>%
  arrange(padj) %>%
  head(15)

p_volcano <- ggplot(volcano_df, aes(x = log2FoldChange, y = neg_log10_padj)) +
  geom_point(aes(color = regulation), alpha = 0.75, size = 1.8) +
  geom_vline(xintercept = c(-1, 1), linetype = "dashed", linewidth = 0.4) +
  geom_hline(yintercept = -log10(0.05), linetype = "dashed", linewidth = 0.4) +
  geom_text_repel(
    data = label_df,
    aes(label = gene),
    size = 3,
    max.overlaps = 30
  ) +
  theme_bw(base_size = 13) +
  labs(
    title = "Volcano plot: disease vs control",
    x = "log2 fold change",
    y = "-log10 adjusted P value",
    color = "Regulation"
  ) +
  theme(
    panel.grid = element_blank(),
    plot.title = element_text(hjust = 0.5)
  )

p_volcano

# 保存图片
ggsave(file.path(figure_dir, "05_volcano_disease_vs_control.pdf"), p_volcano, width = 7, height = 6)
ggsave(file.path(figure_dir, "05_volcano_disease_vs_control.png"), p_volcano, width = 7, height = 6, dpi = 300)


## 16. 准备 GO 富集分析输入基因

GO 富集通常对显著差异基因进行。这里将上调基因和下调基因分开分析，因为两者代表相反的生物学方向。

- 上调基因富集：提示 disease 中增强的生物过程；
- 下调基因富集：提示 disease 中减弱的生物过程。


In [ ]:
# 提取 disease 中上调和下调的基因 symbol
up_symbols <- res_df %>%
  filter(regulation == "Up") %>%
  pull(gene)

down_symbols <- res_df %>%
  filter(regulation == "Down") %>%
  pull(gene)

cat("Number of up-regulated genes:", length(up_symbols), "\n")
cat("Number of down-regulated genes:", length(down_symbols), "\n")

head(up_symbols)
head(down_symbols)


## 17. 基因 ID 转换：SYMBOL 到 ENTREZID

`clusterProfiler::enrichGO()` 常用 ENTREZID 作为输入。因此需要将 gene symbol 转换为 ENTREZID。

这里使用 `org.Hs.eg.db` 进行人类基因 ID 转换。


In [ ]:
# 上调基因 SYMBOL -> ENTREZID
up_map <- bitr(
  up_symbols,
  fromType = "SYMBOL",
  toType = "ENTREZID",
  OrgDb = org.Hs.eg.db
)

# 下调基因 SYMBOL -> ENTREZID
down_map <- bitr(
  down_symbols,
  fromType = "SYMBOL",
  toType = "ENTREZID",
  OrgDb = org.Hs.eg.db
)

cat("Mapped up genes:", nrow(up_map), "\n")
cat("Mapped down genes:", nrow(down_map), "\n")

head(up_map)
head(down_map)


## 18. 上调基因的 GO Biological Process 富集分析

`ont = "BP"` 表示分析 GO Biological Process，即生物过程。

常用参数解释：

| 参数 | 含义 |
|---|---|
| `pAdjustMethod = "BH"` | 使用 Benjamini-Hochberg 方法进行多重检验校正 |
| `pvalueCutoff = 0.05` | 原始 P 值筛选阈值 |
| `qvalueCutoff = 0.2` | q value 筛选阈值 |
| `readable = TRUE` | 将 ENTREZID 转回可读的 gene symbol |


In [ ]:
# 如果可映射的基因太少，富集分析可能没有稳定结果
if (nrow(up_map) >= 5) {
  ego_up <- enrichGO(
    gene = unique(up_map$ENTREZID),
    OrgDb = org.Hs.eg.db,
    keyType = "ENTREZID",
    ont = "BP",
    pAdjustMethod = "BH",
    pvalueCutoff = 0.05,
    qvalueCutoff = 0.2,
    readable = TRUE
  )

  ego_up_df <- as.data.frame(ego_up)

  write.table(
    ego_up_df,
    file = file.path(result_dir, "GOBP_enrichment_up_genes.tsv"),
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
  )

  head(ego_up_df)
} else {
  ego_up <- NULL
  ego_up_df <- data.frame()
  warning("Too few mapped up-regulated genes for GO enrichment.")
}


## 19. 上调基因 GOBP 富集结果可视化

dotplot 中常见元素：

- y 轴：富集到的 GO term；
- x 轴：GeneRatio，即输入基因中落入该 term 的比例；
- 点大小：富集基因数量；
- 点颜色：显著性水平。


In [ ]:
if (!is.null(ego_up) && nrow(as.data.frame(ego_up)) > 0) {
  p_go_up <- dotplot(
    ego_up,
    showCategory = 15
  ) +
    ggtitle("GOBP enrichment of up-regulated genes")

  print(p_go_up)

  ggsave(file.path(figure_dir, "06_GOBP_dotplot_up_genes.pdf"), p_go_up, width = 8, height = 6)
  ggsave(file.path(figure_dir, "06_GOBP_dotplot_up_genes.png"), p_go_up, width = 8, height = 6, dpi = 300)
} else {
  message("No significant GOBP terms for up-regulated genes.")
}


## 20. 下调基因的 GO Biological Process 富集分析

这一节对 disease 中下调的基因单独进行 GOBP 富集。下调基因富集到的过程通常表示 disease 组中相对减弱的功能模块。


In [ ]:
if (nrow(down_map) >= 5) {
  ego_down <- enrichGO(
    gene = unique(down_map$ENTREZID),
    OrgDb = org.Hs.eg.db,
    keyType = "ENTREZID",
    ont = "BP",
    pAdjustMethod = "BH",
    pvalueCutoff = 0.05,
    qvalueCutoff = 0.2,
    readable = TRUE
  )

  ego_down_df <- as.data.frame(ego_down)

  write.table(
    ego_down_df,
    file = file.path(result_dir, "GOBP_enrichment_down_genes.tsv"),
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
  )

  head(ego_down_df)
} else {
  ego_down <- NULL
  ego_down_df <- data.frame()
  warning("Too few mapped down-regulated genes for GO enrichment.")
}


## 21. 下调基因 GOBP 富集结果可视化

该图展示 disease 中下调基因对应的生物过程富集结果。教学时可以和上调基因富集图进行对比，强调“方向性解释”的重要性。


In [ ]:
if (!is.null(ego_down) && nrow(as.data.frame(ego_down)) > 0) {
  p_go_down <- dotplot(
    ego_down,
    showCategory = 15
  ) +
    ggtitle("GOBP enrichment of down-regulated genes")

  print(p_go_down)

  ggsave(file.path(figure_dir, "07_GOBP_dotplot_down_genes.pdf"), p_go_down, width = 8, height = 6)
  ggsave(file.path(figure_dir, "07_GOBP_dotplot_down_genes.png"), p_go_down, width = 8, height = 6, dpi = 300)
} else {
  message("No significant GOBP terms for down-regulated genes.")
}


## 22. 输出文件汇总

最后列出本 Notebook 生成的结果文件和图片文件，方便检查是否成功运行完整流程。


In [ ]:
cat("Result files:\n")
print(list.files(result_dir, full.names = TRUE))

cat("\nFigure files:\n")
print(list.files(figure_dir, full.names = TRUE))

cat("\nAnalysis finished.\n")


## 23. 结果解读建议

完成运行后，可以按以下顺序讲解结果：

1. `sample_qc_summary.tsv`：检查各样本总 counts 和检出基因数；
2. `01_library_size.png` 和 `02_detected_genes.png`：观察样本数据量是否均衡；
3. `03_vst_expression_boxplot.png`：检查标准化后表达分布是否一致；
4. `04_PCA.png`：观察 control 和 disease 是否在整体表达层面分离；
5. `DESeq2_disease_vs_control_all.tsv`：查看完整差异分析结果；
6. `DESeq2_disease_vs_control_DEG.tsv`：查看显著差异基因；
7. `05_volcano_disease_vs_control.png`：展示差异基因整体分布；
8. `GOBP_enrichment_up_genes.tsv` 和 `GOBP_enrichment_down_genes.tsv`：解释疾病组增强或减弱的生物过程。
